# Neural Distribution Transformation — Colab demo

Monotone spline networks for the quantile coupling `T = Q_Y o F_X`.

This notebook installs the package, fits a monotone network, runs the experiment suite (fidelity, convergence, confidence-band coverage, parallel fitting, GPU evaluation) and displays the figures.

**Runtime:** select *Runtime > Change runtime type > GPU* to exercise the CUDA path. Everything also runs on CPU.

## 1. Install

Clones the repository and installs it with the optional extras (`torch`, `joblib`, `matplotlib`, `pandas`).

If the repository is already cloned (for example because you opened this notebook from it), the clone step is skipped and the local copy is installed.

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/yamenetoo/neural-distribution-transformation.git'
WORKDIR = '/content/ndt'

if not os.path.isdir(WORKDIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORKDIR], check=False)

if os.path.isdir(WORKDIR):
    os.chdir(WORKDIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[full]'], check=False)
print('working directory:', os.getcwd())

In [ ]:
import json
from ndt.parallel import describe_environment

print(json.dumps(describe_environment(), indent=2))

## 2. Fit one network

The monotone spline network is

`N(x) = c + sum_j s_j [ relu(x - t_j) - relu(x - t_{j+1}) ]`, with `s_j >= 0`.

Because every weight is non-negative the network is non-decreasing for *any* parameter value, and for fixed knots the fit is a convex non-negative least-squares problem. Here the knots sit at quantiles of the training inputs and the target is the closed-form map.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ndt import fit_msn, get_task, sup_error, ks_distance

task = get_task('P1')                      # N(0,1) -> LogNormal(0,1), T(x) = exp(x)
u = np.arange(1, 4001) / 4001
x_train, y_train = task.ppf_x(u), task.ppf_y(u)

model = fit_msn(x_train, y_train, m=64)
print(model)

lo, hi = task.range_on(0.01, 0.99)
xs = np.linspace(lo, hi, 2000)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, task.T(xs), lw=2, label='true T')
ax.plot(xs, model.predict(xs), '--', lw=2, label='MSN, m=64')
ax.set_xlabel('x')
ax.set_ylabel('T(x)')
ax.set_title('Quantile coupling, Gaussian to log-normal')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print('sup error  :', sup_error(model, task, lo, hi))
print('monotone   :', model.is_monotone(np.linspace(lo, hi, 5001)))
print('KS of N(X) :', ks_distance(model.predict(task.sample_x(np.random.default_rng(0), 100000)), task.cdf_y))

## 3. Fidelity: how the uniform error falls with the number of knots

Compares the fitted network against the true map on eight transport problems, at several knot counts, and reports the Kolmogorov–Smirnov distance next to the Monte Carlo noise floor (the KS distance of the *exact* map on the same draws).

In [ ]:
import pandas as pd
from ndt.experiments import Config, EXPERIMENTS

cfg = Config(quick=True, jobs=2, device='cpu')
fid = pd.DataFrame(EXPERIMENTS['fidelity'](cfg))

cols = ['task', 'm', 'sup', 'mse', 'ks', 'ks_floor', 'monotone']
print(fid[cols].to_string(index=False))

## 4. Convergence from two unpaired samples

Two independent samples of size `n` each, empirical quantile functions, one network per replication. The slope of `log(sup error)` on `log(n)` should be about -0.5.

In [ ]:
conv = pd.DataFrame(EXPERIMENTS['convergence'](cfg))

print(conv[conv.experiment == 'convergence'][['task', 'n', 'mean', 'sd']].to_string(index=False))
print()
print(conv[conv.experiment == 'convergence_slope'][['task', 'slope', 'slope_lo', 'slope_hi']].to_string(index=False))

## 5. Distribution-free confidence band

The band needs only the two empirical distribution functions and the DKW–Massart inequality — no density, continuity or moment assumptions:

`Q_Yhat(F_hat(x) - t_X - t_Y) <= T(x) <= Q_Yhat(F_hat(x) + t_X + t_Y)`

with `t = sqrt(log(4/alpha) / (2n))` for each sample. Below: one realised band, and its empirical coverage against the truth.

In [ ]:
from ndt import band_covers, plugin_band

task = get_task('N1')
rng = np.random.default_rng(0)
xs = task.sample_x(rng, 2000)
ys = task.sample_y(rng, 2000)

grid = np.linspace(*task.range_on(0.05, 0.95), 300)
lo_b, hi_b, radii = plugin_band(grid, xs, ys, alpha=0.05)
covered, all_covered = band_covers((lo_b, hi_b, radii), task.T(grid))

fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(grid, lo_b, hi_b, alpha=0.3, label='95% band')
ax.plot(grid, task.T(grid), 'k', lw=2, label='true T')
ax.set_xlabel('x')
ax.set_title('Distribution-free band, n = 2000')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

print('radii t_X, t_Y    :', radii)
print('pointwise coverage:', round(float(np.mean(covered)), 4))
print('simultaneous      :', all_covered)

## 6. Coverage study

Repeats the construction many times and checks the empirical coverage against the nominal 1 - alpha, pointwise and simultaneously.

In [ ]:
cov = pd.DataFrame(EXPERIMENTS['coverage'](cfg))
print(cov[['task', 'n', 'point_coverage', 'simultaneous_rate', 'median_width']].to_string(index=False))

## 7. GPU acceleration

Fitting is convex NNLS and stays on the CPU. Evaluation and metric computation are what the GPU accelerates, which is where the experiment loops spend their time. On a CPU runtime this cell simply reports the CPU numbers.

In [ ]:
import time
from ndt import resolve_device, torch_predict

device = resolve_device('auto')
print('resolved device:', device)

x_big = np.linspace(-2.3, 2.3, 5_000_000)

t0 = time.perf_counter(); _ = model.predict(x_big); t_cpu = time.perf_counter() - t0
print('numpy  (cpu)   : %.3f s' % t_cpu)

try:
    torch_predict(model, x_big[:1000], device=device)   # warm up
    t0 = time.perf_counter(); _ = torch_predict(model, x_big, device=device); t_dev = time.perf_counter() - t0
    print('torch  (%s): %.3f s' % (device, t_dev))
    print('speed-up       : %.1fx' % (t_cpu / t_dev))
except ImportError:
    print('torch not installed; pip install torch')

## 8. Full suite from the command line

Writes one CSV per experiment into `results/`, plus `environment.json`, `summary.json` and the figure set. Use `--jobs` for parallel replications and `--device cuda` for the GPU path.

In [ ]:
!python -m ndt run --experiments all --out results --jobs 2 --device auto --figures
!du -h results/*.csv

In [ ]:
from IPython.display import Image, display

for name in ['fidelity', 'convergence', 'coverage', 'scaling']:
    path = os.path.join('results', 'figures', name + '.png')
    if os.path.exists(path):
        print(name)
        display(Image(path))

## 9. Notes

- The seed is fixed in `ndt.experiments.common.Config`, so accuracy numbers are reproducible; only the timing experiment varies between runs.
- Every experiment is also importable:

```python
from ndt.experiments import Config, EXPERIMENTS
rows = EXPERIMENTS['coverage'](Config(quick=False, jobs=8, device='cuda'))
```

- The manuscript is *not* in this repository; this package reproduces its numerical section and extends it (knot ablation, band coverage, atom targets, scaling).